In this script, the neighbours are determined for each petrol station based on the original data set using the travel time.

# 1. Import Data

In [57]:
# Global Import and Directories
import sys
sys.path.append("..")
import Setup
from Setup import *

# 2. Export potential Neighbours

The maximum limit of 22 km is used to reduce the number of potential neighbours in a first step, as the subsequent step – calculating the travel times between two petrol stations – is more computationally complex. The distance of 22 km corresponds to the maximum distance that can be covered within 10 minutes (assumption: 130 km/h, maximum speed in Germany).

In [58]:
def km_to_radians(km: float) -> float:
    return km / 6371.0088

def build_candidates_radius(lat_lon_deg: np.ndarray, radius_km: float = 22.0, max_candidates: int = 30):
    lat_lon_rad = np.radians(lat_lon_deg)
    tree = BallTree(lat_lon_rad, metric="haversine")

    r = km_to_radians(radius_km)
    ind, dist = tree.query_radius(lat_lon_rad, r=r, return_distance=True, sort_results=True)

    candidates, cand_dists_km = [], []
    for i in range(lat_lon_deg.shape[0]):
        neigh, d = ind[i], dist[i]

        # self raus
        if len(neigh) > 0 and neigh[0] == i:
            neigh, d = neigh[1:], d[1:]

        # cap
        if max_candidates is not None and len(neigh) > max_candidates:
            neigh, d = neigh[:max_candidates], d[:max_candidates]

        candidates.append(neigh.astype(np.int32))
        cand_dists_km.append((d * 6371.0088).astype(np.float32))

    return candidates, cand_dists_km

def candidates_to_edgelist_uuid(stations_df, lat_lon_deg, candidates, cand_dists_km):
    src_idx_chunks, dst_idx_chunks, dist_chunks = [], [], []

    for src, (dsts, dists) in enumerate(zip(candidates, cand_dists_km)):
        if len(dsts) == 0:
            continue
        src_idx_chunks.append(np.full(len(dsts), src, dtype=np.int32))
        dst_idx_chunks.append(dsts.astype(np.int32))
        dist_chunks.append(dists.astype(np.float32))

    src_idx = np.concatenate(src_idx_chunks) if src_idx_chunks else np.array([], dtype=np.int32)
    dst_idx = np.concatenate(dst_idx_chunks) if dst_idx_chunks else np.array([], dtype=np.int32)
    hav_km = np.concatenate(dist_chunks) if dist_chunks else np.array([], dtype=np.float32)

    src_uuid = stations_df.loc[src_idx, "station_uuid"].to_numpy()
    dst_uuid = stations_df.loc[dst_idx, "station_uuid"].to_numpy()

    return pd.DataFrame({
        "src_uuid": src_uuid,
        "dst_uuid": dst_uuid,
        "haversine_km": hav_km,
        "src_lat": lat_lon_deg[src_idx, 0].astype(np.float32),
        "src_lon": lat_lon_deg[src_idx, 1].astype(np.float32),
        "dst_lat": lat_lon_deg[dst_idx, 0].astype(np.float32),
        "dst_lon": lat_lon_deg[dst_idx, 1].astype(np.float32),
    })

## 2.1. Thuringia

In [4]:
#Load main data set
df_thuringia = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "final_df.csv",
    dtype={9: str}
)

cols = ["station_uuid", "latitude", "longitude", "in_thuringia"]
stations_thuringia = (
    df_thuringia[cols]
    .dropna(subset=["latitude", "longitude", "in_thuringia"])
    .drop_duplicates(subset=["station_uuid"])
    .reset_index(drop=True)
)
stations_thuringia.shape

(1033, 4)

In [9]:
# Create NumPy-Array
lat_lon_deg_thuringia = stations_thuringia[["latitude", "longitude"]].to_numpy(dtype=np.float64)  # (lat, lon)

candidates, cand_dists_km = build_candidates_radius(lat_lon_deg_thuringia, radius_km=22.0, max_candidates=30)
edges = candidates_to_edgelist_uuid(stations_thuringia, lat_lon_deg_thuringia, candidates, cand_dists_km)
edges.shape, edges.head(5)

((30098, 7),
                                src_uuid                              dst_uuid  \
 0  00060065-7890-4444-8888-acdc00000004  e586f8db-b60a-44e5-93eb-63790f76c829   
 1  00060065-7890-4444-8888-acdc00000004  03aff575-0ff9-4816-9478-43f85469fdc6   
 2  00060065-7890-4444-8888-acdc00000004  5525b741-079a-4705-8aef-4948b32b77c6   
 3  00060065-7890-4444-8888-acdc00000004  e41f2c48-d414-4e95-ab8e-60fdd828dac5   
 4  00060065-7890-4444-8888-acdc00000004  ddb01c83-ad26-4e8e-87a8-870acfc599cd   
 
    haversine_km    src_lat    src_lon    dst_lat    dst_lon  
 0      1.062471  50.139297  11.076007  50.136761  11.061634  
 1      1.283435  50.139297  11.076007  50.150173  11.082035  
 2      1.579414  50.139297  11.076007  50.141899  11.054220  
 3      1.769471  50.139297  11.076007  50.151047  11.059260  
 4      2.056437  50.139297  11.076007  50.153999  11.058500  )

In [13]:
out_path = str(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_candidates_r14_max30_Thuringia.csv.gz"
)
edges.to_csv(out_path, index=False, compression="gzip")
out_path

'/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/01_OSMR/stations_candidates_r14_max30.csv.gz'

## 2.2. Hesse

In [60]:
#Load main data set
df_hesse = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Hesse.csv",
    dtype={9: str}
)

cols = ["station_uuid", "latitude", "longitude", "in_hesse"]
stations_hesse = (
    df_hesse[cols]
    .dropna(subset=["latitude", "longitude", "in_hesse"])
    .drop_duplicates(subset=["station_uuid"])
    .reset_index(drop=True)
)
stations_hesse.shape

/var/tmp/pbs.17055445.hpc-batch/ipykernel_51386/67098421.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hesse = pd.read_csv(


(2133, 4)

In [66]:
# Create NumPy-Array
lat_lon_deg_hesse = stations_hesse[["latitude", "longitude"]].to_numpy(dtype=np.float64)  # (lat, lon)

candidates, cand_dists_km = build_candidates_radius(lat_lon_deg_hesse, radius_km=22.0, max_candidates=30)
edges = candidates_to_edgelist_uuid(stations_hesse, lat_lon_deg_hesse, candidates, cand_dists_km)
edges.shape, edges.head(5)

((62879, 7),
                                src_uuid                              dst_uuid  \
 0  00060029-0003-4444-8888-acdc00000003  86da1142-acb5-48a8-bc48-50b74e7337df   
 1  00060029-0003-4444-8888-acdc00000003  54f6febc-11e6-4533-afaa-5aa1ed51c402   
 2  00060029-0003-4444-8888-acdc00000003  f236f82f-6a29-4d6b-aef5-66c9f4454045   
 3  00060029-0003-4444-8888-acdc00000003  00060613-0004-4444-8888-acdc00000004   
 4  00060029-0003-4444-8888-acdc00000003  5c8ee398-0729-4178-a00c-8dde5569e551   
 
    haversine_km    src_lat   src_lon    dst_lat   dst_lon  
 0      0.735133  50.485901  9.717605  50.488438  9.708009  
 1      2.239208  50.485901  9.717605  50.497940  9.692232  
 2      3.641212  50.485901  9.717605  50.468990  9.673540  
 3      4.788028  50.485901  9.717605  50.527729  9.733670  
 4      5.667821  50.485901  9.717605  50.529839  9.676978  )

In [67]:
out_path = str(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_candidates_r14_max30_Hesse.csv.gz"
)
edges.to_csv(out_path, index=False, compression="gzip")
out_path

'/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/01_OSMR/stations_candidates_r14_max30_Hesse.csv.gz'

# 3. Import Travel Times via OSRM

Travel times are calculated on a separate (local) server. For this purpose, a local OSRM routing server was set up, which works with current (non-dynamic) map data from Germany and can determine the travel time between two coordinates. Travel times are calculated in route_candidates.py. The file is then imported again.

## 3.1. Thuringia

In [21]:
tt_thuringia = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_travel_times_thuringia.csv.gz"
)
tt_thuringia["travel_time_min"].describe()

count    28985.000000
mean        16.706708
std          7.715090
min          0.013333
25%         10.861667
50%         16.803333
75%         22.185000
max         43.615000
Name: travel_time_min, dtype: float64

In [22]:
# Test for missing travel times (expected: 0)
tt_thuringia.isna().sum()

src_uuid           0
dst_uuid           0
travel_time_min    0
dtype: int64

In [19]:
tt_thuringia.head(5)

,src_uuid,dst_uuid,travel_time_min
0,00060065-7890-4444-8888-acdc00000004,e586f8db-b60a-44e5-93eb-63790f76c829,1.583333
1,00060065-7890-4444-8888-acdc00000004,03aff575-0ff9-4816-9478-43f85469fdc6,2.058333
2,00060065-7890-4444-8888-acdc00000004,5525b741-079a-4705-8aef-4948b32b77c6,2.758333
3,00060065-7890-4444-8888-acdc00000004,e41f2c48-d414-4e95-ab8e-60fdd828dac5,4.403333
4,00060065-7890-4444-8888-acdc00000004,ddb01c83-ad26-4e8e-87a8-870acfc599cd,5.015000


## 3.2. Hesse

In [68]:
tt_hesse = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_travel_times_hesse.csv.gz"
)
tt_hesse["travel_time_min"].describe()

count    62879.000000
mean        13.487690
std          8.373928
min          0.000000
25%          8.022500
50%         12.215000
75%         17.748333
max         82.520000
Name: travel_time_min, dtype: float64

In [69]:
# Test for missing travel times (expected: 0)
tt_hesse.isna().sum()

src_uuid           0
dst_uuid           0
travel_time_min    0
dtype: int64

In [70]:
tt_hesse.head(5)

,src_uuid,dst_uuid,travel_time_min
0,00060029-0003-4444-8888-acdc00000003,86da1142-acb5-48a8-bc48-50b74e7337df,2.050000
1,00060029-0003-4444-8888-acdc00000003,54f6febc-11e6-4533-afaa-5aa1ed51c402,4.431667
2,00060029-0003-4444-8888-acdc00000003,f236f82f-6a29-4d6b-aef5-66c9f4454045,8.191667
3,00060029-0003-4444-8888-acdc00000003,00060613-0004-4444-8888-acdc00000004,11.608334
4,00060029-0003-4444-8888-acdc00000003,5c8ee398-0729-4178-a00c-8dde5569e551,8.321667
